# **Collecting data by vehicles**

In [1]:
SIMULATION_STABLE = "../02_scenario/hats.sumocfg"
SEEDS = ['42', '1234', '1867', '613', '1001', '704', '882', '405', '269', '120']

SENSOR_RANGE = 20 #[m]

VEHICLES_PATH = "../../02_data/veh_list.json"
OUTPUT_DIR = "../../02_data/01_simulation_results/"

In [2]:
import json
import numpy as np
from tqdm.notebook import trange

import pandas as pd

In [3]:
import os, sys
import traci

SUMO_HOME = os.environ["SUMO_HOME"] #locating the simulator
sys.path.append(SUMO_HOME+"/tools")

import sumolib
from importlib import reload
import traci.constants as tc

In [4]:
with open(VEHICLES_PATH) as f:
    vehicle_dicts = json.load(f)
    vehicles = vehicle_dicts["train_vehs"] + vehicle_dicts["test_vehs"]

In [7]:
def run_simulation(seed):   
    #running stable simulation and obtaining results by TraCI:
    sumo_cmd = ["sumo", "-c", SIMULATION_STABLE, "--seed", str(seed), "--no-step-log"]
    traci.start(sumo_cmd)

    e_ids = traci.parkingarea.getIDList()

    vehicle_data = pd.DataFrame(columns=["meas_id", "veh_id", "edge_id", "time", "avg_speed"])
    active_vehicles = set()
    speed_measurement_list = []
    vehicle_positions = []

    end_time = traci.simulation.getEndTime()
    timestep = 0
    header_set = False
    while ((end_time > 0) and (timestep < end_time)) or ((end_time<0) and (traci.simulation.getMinExpectedNumber()>0)):
        traci.simulationStep()
        timestep = traci.simulation.getTime()
                
        #new cars are added:
        new_vehicles = traci.simulation.getDepartedIDList()
        for veh in new_vehicles:
            if veh in vehicles:
                active_vehicles.add(veh)
                try:
                    traci.vehicle.subscribeContext(veh, tc.CMD_GET_EDGE_VARIABLE, SENSOR_RANGE, [tc.LAST_STEP_MEAN_SPEED])
                except:
                    pass
        #stop ending (restarting) vehicles:
        starting_vehicles = traci.simulation.getStopEndingVehiclesIDList()
        for veh in starting_vehicles:
            active_vehicles.add(veh)
            try:
                traci.vehicle.subscribeContext(veh, tc.CMD_GET_EDGE_VARIABLE, SENSOR_RANGE, [tc.LAST_STEP_MEAN_SPEED])
            except:
                pass
        #stopping vehicles:
        stopping_vehicles = traci.simulation.getStopStartingVehiclesIDList()
        for veh in stopping_vehicles:
            if veh in active_vehicles:
                active_vehicles.remove(veh)
                try:
                    traci.vehicle.unsubscribeContext(veh, tc.CMD_GET_EDGE_VARIABLE, SENSOR_RANGE)
                except:
                    pass
        #arriving vehicles:
        arriving_vehicles = traci.simulation.getArrivedIDList()
        for veh in arriving_vehicles:
            if veh in active_vehicles:
                active_vehicles.remove(veh)

        edge_speeds = {}
        for e_id in e_ids:
            edge_speeds[e_id] = traci.edge.getLastStepMeanSpeed(e_id)

        for veh in active_vehicles:
            #parking data:
            measured_edges = traci.vehicle.getContextSubscriptionResults(veh)

            for e in measured_edges:
                mean_speed = measured_edges[e][tc.LAST_STEP_MEAN_SPEED]

                new_speed_measurement = {"veh_id": veh,
                                    "edge_id": e,
                                    "time": timestep,
                                    "avg_speed": mean_speed}
                if e[0]!=":":
                    speed_measurement_list.append(new_speed_measurement)
            
            #vehicle positions:
            road = traci.vehicle.getRoadID(veh)
            new_position_measurement = {
                "veh_id": veh,
                "time": timestep,
                "edge": road
            }
            vehicle_positions.append(new_position_measurement)
            
        if timestep%600 == 0:
            vehicle_speed_data = pd.DataFrame.from_records(speed_measurement_list)
            vehicle_speed_data = vehicle_speed_data.drop_duplicates()
            vehicle_speed_data.to_csv(f'{OUTPUT_DIR}/pspeeds_by_vehs_{seed}.csv', mode="a" if header_set else "w",
                                      header=not(header_set), index=False)

            vehicle_position_data = pd.DataFrame.from_records(vehicle_positions)
            vehicle_position_data = vehicle_position_data.drop_duplicates()
            vehicle_position_data.to_csv(f"{OUTPUT_DIR}/vehicle_positions_{seed}.csv", mode="a" if header_set else "w",
                                         header=not(header_set), index=False)
            
            header_set = True
            
            speed_measurement_list = []
            vehicle_positions = []

    traci.close()
    vehicle_speed_data = pd.DataFrame.from_records(speed_measurement_list)
    vehicle_speed_data = vehicle_speed_data.drop_duplicates()
    vehicle_speed_data.to_csv(f'{OUTPUT_DIR}/pspeeds_by_vehs_{seed}.csv', mode='a', header=False, index=False)

    vehicle_position_data = pd.DataFrame.from_records(vehicle_positions)
    vehicle_position_data = vehicle_position_data.drop_duplicates()
    vehicle_position_data.to_csv(f"{OUTPUT_DIR}/vehicle_positions_{seed}.csv", mode='a', header=False, index=False)
    

In [8]:
from multiprocessing import Pool

#simulation in parallel:
with Pool(len(SEEDS)) as pool:
    pool.map(run_simulation, SEEDS)

 Retrying in 1 seconds Retrying in 1 seconds Retrying in 1 seconds Retrying in 1 seconds Retrying in 1 seconds Retrying in 1 seconds Retrying in 1 seconds Retrying in 1 seconds Retrying in 1 seconds


 Retrying in 1 seconds






***Starting server on port 39405 ***
***Starting server on port 60017 ***
***Starting server on port 53487 ***
***Starting server on port 51125 ***
***Starting server on port 38931 ***
***Starting server on port 57941 ***
***Starting server on port 35017 ***
***Starting server on port 34051 ***
***Starting server on port 60749 ***
***Starting server on port 37017 ***
Loading net-file from '../02_scenario/hats.net.xml' ...Loading net-file from '../02_scenario/hats.net.xml' ...Loading net-file from '../02_scenario/hats.net.xml' ...Loading net-file from '../02_scenario/hats.net.xml' ...Loading net-file from '../02_scenario/hats.net.xml' ...Loading net-file from '../02_scenario/hats.net.xml' ...Loading net-file from '../02_scenario/hats.net.xml' ...Loading net-fil

Simulation ended at time: 15120.00.
Reason: TraCI requested termination.
Performance:
 Duration: 944.80s
 TraCI-Duration: 520.08s
 Real time factor: 16.0035
 UPS: 14621.865061
Vehicles:
 Inserted: 25290 (Loaded: 26132)
 Running: 886
 Waiting: 1
 Teleports: 17 (Jam: 8, Yield: 9)
 Emergency Braking: 13
Statistics (avg of 24404):
 RouteLength: 3652.84
 Speed: 7.28
 Duration: 553.75
 WaitingTime: 168.46
 TimeLoss: 262.10
 DepartDelay: 2.58
DijkstraRouter answered 15037 queries and explored 438.86 edges on average.
DijkstraRouter spent 5.00s answering queries (0.33ms on average).


Simulation ended at time: 15120.00.
Reason: TraCI requested termination.
Performance:
 Duration: 966.87s
 TraCI-Duration: 530.01s
 Real time factor: 15.638
 UPS: 14674.266424
Vehicles:
 Inserted: 25284 (Loaded: 26132)
 Running: 915
 Waiting: 1
 Teleports: 26 (Jam: 15, Yield: 11)
 Emergency Braking: 11
Statistics (avg of 24369):
 RouteLength: 3655.24
 Speed: 7.20
 Duration: 566.66
 WaitingTime: 177.97
 TimeLoss: 274.30
 DepartDelay: 2.60
DijkstraRouter answered 15143 queries and explored 437.93 edges on average.
DijkstraRouter spent 4.99s answering queries (0.33ms on average).
Simulation ended at time: 15120.00.
Reason: TraCI requested termination.
Performance:
 Duration: 969.20s
 TraCI-Duration: 530.44s
 Real time factor: 15.6004
 UPS: 14668.981627
Vehicles:
 Inserted: 25259 (Loaded: 26132)
 Running: 943
 Waiting: 1
 Teleports: 27 (Collisions: 1, Jam: 16, Yield: 11)
 Emergency Braking: 5
Statistics (avg of 24316):
 RouteLength: 3654.23
 Speed: 7.20
 Duration: 567.58
 WaitingTime: 179.9

Simulation ended at time: 15120.00.
Reason: TraCI requested termination.
Performance:
 Duration: 990.02s
 TraCI-Duration: 543.11s
 Real time factor: 15.2724
 UPS: 14702.932974
Vehicles:
 Inserted: 25239 (Loaded: 26132)
 Running: 876
 Waiting: 4
 Teleports: 36 (Jam: 21, Yield: 15)
 Emergency Braking: 13
Statistics (avg of 24363):
 RouteLength: 3662.65
 Speed: 7.07
 Duration: 585.57
 WaitingTime: 192.70
 TimeLoss: 292.85
 DepartDelay: 2.99
DijkstraRouter answered 15619 queries and explored 447.01 edges on average.
DijkstraRouter spent 5.22s answering queries (0.33ms on average).


Simulation ended at time: 15120.00.
Reason: TraCI requested termination.
Performance:
 Duration: 1009.12s
 TraCI-Duration: 551.99s
 Real time factor: 14.9834
 UPS: 14730.751279
Vehicles:
 Inserted: 25216 (Loaded: 26132)
 Running: 903
 Waiting: 1
 Teleports: 29 (Jam: 15, Yield: 14)
 Emergency Braking: 8
Statistics (avg of 24313):
 RouteLength: 3663.84
 Speed: 7.04
 Duration: 597.90
 WaitingTime: 203.23
 TimeLoss: 305.05
 DepartDelay: 2.92
DijkstraRouter answered 15456 queries and explored 444.77 edges on average.
DijkstraRouter spent 5.10s answering queries (0.33ms on average).
Simulation ended at time: 15120.00.
Reason: TraCI requested termination.
Performance:
 Duration: 1016.64s
 TraCI-Duration: 556.81s
 Real time factor: 14.8726
 UPS: 14878.970686
Vehicles:
 Inserted: 25229 (Loaded: 26132)
 Running: 957
 Waiting: 1
 Teleports: 43 (Collisions: 1, Jam: 26, Yield: 15, Wrong Lane: 2)
 Emergency Braking: 10
Statistics (avg of 24272):
 RouteLength: 3669.66
 Speed: 7.01
 Duration: 605.14
 

Simulation ended at time: 15120.00.
Reason: TraCI requested termination.
Performance:
 Duration: 1067.13s
 TraCI-Duration: 580.53s
 Real time factor: 14.1689
 UPS: 14955.785019
Vehicles:
 Inserted: 25114 (Loaded: 26132)
 Running: 1029
 Waiting: 5
 Teleports: 83 (Collisions: 1, Jam: 45, Yield: 36, Wrong Lane: 2)
 Emergency Braking: 16
Statistics (avg of 24085):
 RouteLength: 3666.67
 Speed: 6.96
 Duration: 639.78
 WaitingTime: 237.53
 TimeLoss: 346.49
 DepartDelay: 2.98
DijkstraRouter answered 15591 queries and explored 445.98 edges on average.
DijkstraRouter spent 5.01s answering queries (0.32ms on average).
